# Best-of-N speed benchmark: native vs prompt-duplication

Compare `bon_search_v1.best_of_n_v1` (vLLM's native n-sampling)
against `bon_search_v1.best_of_n_v2` (prompt duplicated `config.n`
times, `n=1`) on the same model and prompts. Times `num_trials`
runs per variant with one untimed warmup.

Useful for picking the right vLLM batching idiom at your
`config.n` and prompt-count regime.

## Setup

In [ ]:
import os
os.environ["VLLM_CONFIGURE_LOGGING"] = "0"

import logging
logging.basicConfig(format='%(message)s', level=logging.FATAL + 1)

import warnings
warnings.filterwarnings("ignore")


import sys
sys.path.append("..")

import gc
import statistics
import time

import torch
from vllm import LLM

from sal.config import Config

from core import bon_search_v1
from utils.load_data import load_data_hf

In [ ]:
# Dataset and model paths
base_dir = '/groups/chichengz/tnn/datasets'

ds_name = "prm800k"
ds_split = "test"
ds_dir = os.path.join(base_dir, "prm800k/math_splits")

# llm_dir = os.path.join(
#     base_dir, "Llama-3.2-1B-Instruct-GGUF/Llama-3.2-1B-Instruct.Q4_K_M.gguf"
# )
llm_dir = os.path.join(base_dir, "Llama3.2-1B-Instruct")
prm_dir = os.path.join(base_dir, "Llama3.1-8B-PRM-Deepseek-Data")

In [ ]:
# Best-of-N search params
config = Config()
config.agg_strategy = 'last'
config.temperature = 0.8
config.max_tokens = 2048
config.n = 256
config.filter_duplicates = True
config.date_string = "Aug 1 2025"
config.seed = 0

# Benchmark knobs
level = 4                          # MATH difficulty level
MAX_QUESTIONS = None               # None = full level slice; int caps it
num_trials = 2                     # timed runs per variant
warmup = 1                         # untimed warmup runs per variant
llm_gpu_memory_utilization = 0.7

## Load model

In [ ]:
# enforce_eager=True disables CUDA graphs - skips cudagraph capture cost
# at load, giving more stable latency at small num_trials.
llm_vllm = LLM(
    model=llm_dir,
    tensor_parallel_size=1,
    max_model_len=5000,
    gpu_memory_utilization=llm_gpu_memory_utilization,
    enforce_eager=True,
    distributed_executor_backend=None,
    dtype="float16",
    seed=config.seed,
)

free, total = torch.cuda.mem_get_info(0)
print(f'GPU memory used: {(total - free) / (1024**3):.2f} GB')

In [ ]:
dataset = load_data_hf(ds_dir, ds_split=ds_split, level=level)

num_questions = len(dataset)
if MAX_QUESTIONS is not None:
    num_questions = min(num_questions, MAX_QUESTIONS)
batch_of_questions = [dataset[i]['question'] for i in range(num_questions)]
print(f"num_questions = {num_questions}")

## Compare variants

- **v1 (native)**: single prompt, `n=config.n` — vLLM generates
  all completions internally for each prompt.
- **v2 (prompt_dup)**: duplicate each prompt `config.n` times,
  `n=1` — continuous batching across the duplicated inputs.

In [ ]:
methods = [
    ('native     (v1)', bon_search_v1.best_of_n_v1),
    ('prompt_dup (v2)', bon_search_v1.best_of_n_v2),
]

results = []
for name, method in methods:
    print(f"\n=== {name} ===")

    # Warmup (untimed) - absorbs first-call init inside the method
    for w in range(warmup):
        method(batch_of_questions, config, llm_vllm, 10_000 + w)

    times = []
    for trial_idx in range(num_trials):
        start = time.perf_counter()
        method(batch_of_questions, config, llm_vllm, trial_idx)
        elapsed = time.perf_counter() - start
        times.append(elapsed)
        print(
            f"  trial {trial_idx}: {elapsed:>7.2f}s total, "
            f"{elapsed / num_questions:.4f}s/question"
        )

    results.append((name, times))

## Summary

In [ ]:
print(
    f"=== Summary (level={level}, "
    f"n_questions={num_questions}, n_trials={num_trials}) ==="
)
header = (
    f"{'variant':<20}{'mean s/trial':>14}{'std':>8}{'s/question':>14}"
)
print(header)
print('-' * len(header))
for name, times in results:
    mean = statistics.mean(times)
    std = statistics.stdev(times) if len(times) > 1 else 0.0
    print(
        f"{name:<20}{mean:>14.2f}{std:>8.2f}{mean/num_questions:>14.4f}"
    )